# 21 — Performance: Profiling, Big‑O, and Optimization

Goal: make Python code faster *the right way* (measure first, optimize last).

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U numpy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Performance rules

1. Make it correct.
2. Make it clear.
3. Make it fast **only after measuring**.

Tools:
- `timeit` for microbenchmarks
- `cProfile` for whole-program profiling
- `tracemalloc` for memory tracking

## 2.
L2: Big-O intuition (practical)

- list membership `x in list` is O(n)
- set/dict membership `x in set/dict` is ~O(1)
- sorting is O(n log n)

Pick data structures first; optimize loops second.

In [ ]:

import timeit

setup = "xs = list(range(10000)); s = set(xs)"
t_list = timeit.timeit("9999 in xs", setup=setup, number=2000)
t_set  = timeit.timeit("9999 in s",  setup=setup, number=2000)
print("list membership:", round(t_list, 4), "sec")
print("set  membership:", round(t_set, 4), "sec")


## 3.
L3: `cProfile` (where time really goes)

Use in scripts:

```bash
python -m cProfile -o profile.out your_script.py
python -m pstats profile.out
```

In notebooks we can demonstrate quickly on a function.

In [ ]:

import cProfile, pstats, io

def work(n: int) -> int:
    s = 0
    for i in range(n):
        s += i*i
    return s

pr = cProfile.Profile()
pr.enable()
work(200_000)
pr.disable()

buf = io.StringIO()
ps = pstats.Stats(pr, stream=buf).sort_stats("tottime")
ps.print_stats(5)
print(buf.getvalue().splitlines()[0])
print("\n".join(buf.getvalue().splitlines()[1:6]))


## 4.
L4: Caching with `lru_cache`

Memoization is often a bigger win than micro-optimizing loops.

In [ ]:

from functools import lru_cache

@lru_cache(maxsize=None)
def choose(n: int, k: int) -> int:
    if k == 0 or k == n:
        return 1
    return choose(n-1, k-1) + choose(n-1, k)

print(choose(30, 15))


## 5.
L5: When to use NumPy (vectorization)

If you’re doing numeric work on large arrays, NumPy is often *orders* of magnitude faster.
(But don’t import NumPy just to sum a small list.)

In [ ]:

try:
    import numpy as np
    a = np.arange(10)
    print((a * a).sum())
except Exception as e:
    print("numpy not installed:", e)


## 6.
L6: Exercises

1. Time list vs set membership on a bigger N.
2. Profile a function of your own and identify the hotspot.
3. Add caching to a recursive function and measure improvement.

## 7.
L7: Memory profiling basics (`tracemalloc`)

Track where allocations come from.

In [ ]:

import tracemalloc

tracemalloc.start()
x = [i for i in range(50_000)]
current, peak = tracemalloc.get_traced_memory()
print("current KB:", current // 1024, "peak KB:", peak // 1024)
tracemalloc.stop()


## 8.
L8: Avoiding common slow patterns

- repeated string concatenation in a loop → use list + join
- `list.pop(0)` → use `collections.deque`
- nested loops over large data → consider algorithmic changes or vectorization